In [6]:
from pathlib import Path
import calendar
import time

import pandas as pd
import requests

In [7]:
RAW_DIR = Path("../data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

LATITUDE = 41.9981
LONGITUDE = 21.4254

START_YEAR = 2016
END_YEAR = 2025

WEATHER_URL = "https://archive-api.open-meteo.com/v1/archive"

In [8]:
ALL_HOURLY_FEATURES = [
    "temperature_2m",
    "relative_humidity_2m",
    "dew_point_2m",
    "apparent_temperature",
    "pressure_msl",
    "surface_pressure",
    "precipitation",
    "rain",
    "snowfall",
    "cloud_cover",
    "cloud_cover_low",
    "cloud_cover_mid",
    "cloud_cover_high",
    "shortwave_radiation",
    "direct_radiation",
    "direct_normal_irradiance",
    "diffuse_radiation",
    "global_tilted_irradiance",
    "sunshine_duration",
    "wind_speed_10m",
    "wind_speed_100m",
    "wind_direction_10m",
    "wind_direction_100m",
    "wind_gusts_10m",
    "et0_fao_evapotranspiration",
    "weather_code",
    "snow_depth",
    "vapour_pressure_deficit",
    "soil_temperature_0_to_7cm",
    "soil_temperature_7_to_28cm",
    "soil_temperature_28_to_100cm",
    "soil_temperature_100_to_255cm",
    "soil_moisture_0_to_7cm",
    "soil_moisture_7_to_28cm",
    "soil_moisture_28_to_100cm",
    "soil_moisture_100_to_255cm",
]

In [11]:
import random

def fetch_weather_year(year, latitude, longitude, features):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": f"{year}-01-01",
        "end_date": f"{year}-12-31",
        "hourly": ",".join(features),
        "timezone": "Europe/Skopje",
        "wind_speed_unit": "ms",
        "tilt": 0,
        "azimuth": 0,
    }

    max_attempts = 6

    for attempt in range(1, max_attempts + 1):
        try:
            response = requests.get(
                WEATHER_URL,
                params=params,
                timeout=(30, 240)
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after and retry_after.isdigit():
                    wait_seconds = int(retry_after)
                else:
                    wait_seconds = min(60 * (2 ** (attempt - 1)), 900)
                    wait_seconds += random.randint(0, 20)

                print(
                    f"{year}: rate limited (429). "
                    f"Waiting {wait_seconds} seconds before retrying..."
                )

                time.sleep(wait_seconds)
                continue

            response.raise_for_status()

            payload = response.json()
            df = pd.DataFrame(payload["hourly"])

            df["time"] = pd.to_datetime(df["time"])
            df["station_latitude"] = latitude
            df["station_longitude"] = longitude
            df["weather_timezone"] = payload["timezone"]

            return df, payload["hourly_units"]

        except requests.exceptions.Timeout:
            wait_seconds = min(30 * attempt, 180)

            print(
                f"{year}: timeout on attempt {attempt}/{max_attempts}. "
                f"Waiting {wait_seconds} seconds..."
            )

            time.sleep(wait_seconds)

        except requests.exceptions.RequestException as error:
            wait_seconds = min(30 * attempt, 180)

            print(
                f"{year}: request failed on attempt "
                f"{attempt}/{max_attempts}: {error}. "
                f"Waiting {wait_seconds} seconds..."
            )

            time.sleep(wait_seconds)

    raise RuntimeError(
        f"Could not download weather data for {year} "
        f"after {max_attempts} attempts."
    )

In [12]:
all_years = []
hourly_units = None

for year in range(START_YEAR, END_YEAR + 1):
    output_file = RAW_DIR / f"weather_hourly_skopje_{year}.parquet"

    if output_file.exists():
        print(f"{year}: already saved, loading file")
        year_df = pd.read_parquet(output_file)

    else:
        print(f"{year}: downloading all hourly features...")

        year_df, hourly_units = fetch_weather_year(
            year=year,
            latitude=LATITUDE,
            longitude=LONGITUDE,
            features=ALL_HOURLY_FEATURES
        )

        expected_hours = 8784 if calendar.isleap(year) else 8760

        if len(year_df) != expected_hours:
            raise ValueError(
                f"{year}: expected {expected_hours} rows, "
                f"but received {len(year_df)}"
            )

        year_df.to_parquet(output_file, index=False)

        print(f"{year}: saved {len(year_df):,} rows")

        time.sleep(1)

    all_years.append(year_df)

2016: already saved, loading file
2017: already saved, loading file
2018: already saved, loading file
2019: already saved, loading file
2020: already saved, loading file
2021: already saved, loading file
2022: already saved, loading file
2023: downloading all hourly features...
2023: saved 8,760 rows
2024: downloading all hourly features...
2024: saved 8,784 rows
2025: downloading all hourly features...
2025: saved 8,760 rows


In [13]:
weather_hourly_df = pd.concat(all_years, ignore_index=True)

weather_hourly_df = weather_hourly_df.sort_values("time").reset_index(drop=True)

print("Shape:", weather_hourly_df.shape)

display(weather_hourly_df.head())
display(weather_hourly_df.tail())

Shape: (87672, 40)


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,pressure_msl,surface_pressure,precipitation,rain,snowfall,...,soil_temperature_7_to_28cm,soil_temperature_28_to_100cm,soil_temperature_100_to_255cm,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm,soil_moisture_100_to_255cm,station_latitude,station_longitude,weather_timezone
0,2016-01-01 00:00:00,-2.6,38,-14.9,-6.9,1033.0,1000.9,0.0,0.0,0.0,...,1.5,4.9,11.6,0.316,0.321,0.32,0.294,41.9981,21.4254,Europe/Skopje
1,2016-01-01 01:00:00,-2.7,39,-14.8,-6.9,1032.4,1000.3,0.0,0.0,0.0,...,1.5,4.9,11.5,0.316,0.321,0.32,0.294,41.9981,21.4254,Europe/Skopje
2,2016-01-01 02:00:00,-2.9,40,-14.8,-7.0,1031.9,999.8,0.0,0.0,0.0,...,1.5,4.8,11.5,0.316,0.321,0.32,0.294,41.9981,21.4254,Europe/Skopje
3,2016-01-01 03:00:00,-4.0,40,-15.6,-7.9,1031.6,999.3,0.0,0.0,0.0,...,1.5,4.8,11.5,0.316,0.321,0.32,0.294,41.9981,21.4254,Europe/Skopje
4,2016-01-01 04:00:00,-4.1,40,-15.6,-8.0,1031.3,999.0,0.0,0.0,0.0,...,1.4,4.8,11.5,0.316,0.321,0.32,0.294,41.9981,21.4254,Europe/Skopje


,time,temperature_2m,relative_humidity_2m,dew_point_2m,apparent_temperature,pressure_msl,surface_pressure,precipitation,rain,snowfall,...,soil_temperature_7_to_28cm,soil_temperature_28_to_100cm,soil_temperature_100_to_255cm,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,soil_moisture_28_to_100cm,soil_moisture_100_to_255cm,station_latitude,station_longitude,weather_timezone
87667,2025-12-31 19:00:00,-2.5,44,-13.2,-6.7,1019.9,988.2,0.0,0.0,0.0,...,0.9,4.5,12.6,0.342,0.356,0.343,0.157,41.9981,21.4254,Europe/Skopje
87668,2025-12-31 20:00:00,-3.0,52,-11.6,-7.1,1020.1,988.3,0.0,0.0,0.0,...,0.9,4.5,12.6,0.342,0.356,0.343,0.157,41.9981,21.4254,Europe/Skopje
87669,2025-12-31 21:00:00,-2.9,53,-11.3,-6.9,1020.2,988.4,0.0,0.0,0.0,...,0.9,4.5,12.6,0.342,0.355,0.343,0.157,41.9981,21.4254,Europe/Skopje
87670,2025-12-31 22:00:00,-3.4,60,-10.1,-7.1,1020.4,988.6,0.0,0.0,0.0,...,0.9,4.5,12.6,0.342,0.355,0.343,0.157,41.9981,21.4254,Europe/Skopje
87671,2025-12-31 23:00:00,-3.4,58,-10.4,-7.0,1021.1,989.3,0.0,0.0,0.0,...,0.9,4.5,12.6,0.343,0.355,0.343,0.157,41.9981,21.4254,Europe/Skopje


In [14]:
expected_columns = ["time"] + ALL_HOURLY_FEATURES

missing_columns = [
    column
    for column in expected_columns
    if column not in weather_hourly_df.columns
]

print("Missing requested columns:", missing_columns)
print("Number of rows:", len(weather_hourly_df))
print("Number of columns:", len(weather_hourly_df.columns))

Missing requested columns: []
Number of rows: 87672
Number of columns: 40


In [15]:
missing_report = pd.DataFrame({
    "data_type": weather_hourly_df.dtypes.astype(str),
    "missing_count": weather_hourly_df.isna().sum(),
    "missing_percent": (
        weather_hourly_df.isna().mean() * 100
    ).round(2)
})

missing_report = missing_report.sort_values(
    "missing_percent",
    ascending=False
)

display(missing_report)

,data_type,missing_count,missing_percent
time,datetime64[us],0,0.0
temperature_2m,float64,0,0.0
relative_humidity_2m,int64,0,0.0
dew_point_2m,float64,0,0.0
apparent_temperature,float64,0,0.0
pressure_msl,float64,0,0.0
surface_pressure,float64,0,0.0
precipitation,float64,0,0.0
rain,float64,0,0.0
snowfall,float64,0,0.0


In [16]:
combined_file = RAW_DIR / "weather_hourly_skopje_2016_2025.parquet"

weather_hourly_df.to_parquet(combined_file, index=False)

print(f"Saved combined dataset to: {combined_file}")

Saved combined dataset to: ../data/raw/weather_hourly_skopje_2016_2025.parquet
